# FINAL Step 2.6 — Original-Image Patch Paper-Style Investigation

Supervisor-suggested sensitivity check: keep the existing preprocessing-derived coordinates/labels, but extract the actual classification patches from the original INbreast DICOM images before pectoral removal, breast masking, CLAHE, or percentile clipping.

This notebook is intentionally separate from the main FINAL Step 2 notebooks. It does **not** modify any current notebook or report file. The aim is to test whether the paper-style WS/CNN/fusion pipeline improves when the sampled tissue/mass coordinates are applied to the raw images.

Important design choices:

- Background tissue uses the same 434 accepted fatty/fibroglandular coordinates from `Step 1.3`.
- Mass uses the same 115 post-quality-filter mass samples as the current paper-style mass notebook, so the comparison isolates pixel source rather than changing the dataset.
- Raw DICOM images are only linearly min-max scaled to `[0, 1]` before cropping/resizing. No pectoral removal, breast/background masking, CLAHE, or percentile clipping is applied.
- The paper-style classifiers and current shared seed protocol are imported from `project34.protocol`.

In [1]:
from pathlib import Path
import json
import math
import os
import random
import warnings

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from kymatio.scattering2d.frontend.numpy_frontend import ScatteringNumPy2D as Scattering2D
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import models

import sys
PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "Step 2 - experiments NOTEBOOKS" else Path.cwd().resolve()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from project34.protocol import SEEDS, SubspaceKNN, set_seed

warnings.filterwarnings("ignore", category=UserWarning)
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

DATA = PROJECT / "data"
OUT = DATA / "outputs" / "original_image_patch_investigation"
OUT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = list(SEEDS)
print("Project:", PROJECT)
print("Output:", OUT)
print("Seeds:", SEEDS)
print("Device:", DEVICE)

Project: /home/nabeel/project34/Project34
Output: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation
Seeds: [1, 7, 34, 42, 99]
Device: cpu


## 1. Build Original-Image Patch Manifests

The preprocessing notebooks already identify which tissue/mass locations should be sampled. Here we reuse those coordinates, map each row back to its original DICOM, and extract a resized patch from a simple full-image min-max version of the DICOM pixel array.

This keeps the geometric selection provided by preprocessing, while avoiding the downstream pixel operations that may remove useful image information.

In [2]:
def resolve_project_path(path_like):
    path = Path(path_like)
    if path.is_absolute():
        return path
    # Existing manifests were written from notebook folders, so ../data/... means PROJECT/data/...
    if path.parts and path.parts[0] == "..":
        return (PROJECT / Path(*path.parts[1:])).resolve()
    return (PROJECT / path).resolve()


def load_dicom_minmax(path):
    """Read original DICOM pixels and apply only a linear full-image min-max scale."""
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)
    lo = float(np.nanmin(img))
    hi = float(np.nanmax(img))
    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)
    return ((img - lo) / (hi - lo)).astype(np.float32)


_dicom_cache = {}
def cached_raw_image(path):
    path = resolve_project_path(path)
    key = str(path)
    if key not in _dicom_cache:
        _dicom_cache[key] = load_dicom_minmax(path)
    return _dicom_cache[key]


def crop_resize_raw(dicom_path, y0, y1, x0, x1, *, inclusive_y1x1=False):
    img = cached_raw_image(dicom_path)
    h, w = img.shape
    y0 = int(max(0, min(h, round(y0))))
    x0 = int(max(0, min(w, round(x0))))
    y1 = int(round(y1)) + (1 if inclusive_y1x1 else 0)
    x1 = int(round(x1)) + (1 if inclusive_y1x1 else 0)
    y1 = int(max(y0 + 1, min(h, y1)))
    x1 = int(max(x0 + 1, min(w, x1)))
    crop = img[y0:y1, x0:x1]
    patch = cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(patch, 0.0, 1.0).astype(np.float32)


preproc = pd.read_csv(DATA / "outputs" / "preprocessed" / "preproc_index.csv")
preproc["final_resolved"] = preproc["final_npy"].map(lambda p: str(resolve_project_path(p)))
preproc["dicom_resolved"] = preproc["dicom"].map(lambda p: str(resolve_project_path(p)))
final_to_dicom = dict(zip(preproc["final_resolved"], preproc["dicom_resolved"]))

patches_index = pd.read_csv(DATA / "outputs" / "background_patches" / "patches_index.csv")
tissue_rows = []
for row in patches_index.itertuples(index=False):
    source_final = str(resolve_project_path(row.source_final_npy))
    tissue_rows.append({
        "sample_id": Path(row.patch_npy).stem,
        "file_id": str(row.file_id),
        "label": row.label,
        "class_id": 1 if row.label == "fibroglandular" else 0,
        "acr": int(row.acr),
        "dicom_path": final_to_dicom[source_final],
        "y0": int(row.y0),
        "y1": int(row.y1),
        "x0": int(row.x0),
        "x1": int(row.x1),
        "orig_size": int(row.orig_size),
    })
tissue_manifest = pd.DataFrame(tissue_rows)

mass_index = pd.read_csv(DATA / "outputs" / "masses" / "mass_index.csv")
current_mass_manifest = pd.read_csv(DATA / "outputs" / "final_mass_replication" / "mass_manifest.csv")
mass_manifest = current_mass_manifest.merge(
    mass_index[["file_id", "roi_index", "y0", "y1", "x0", "x1", "dicom_path", "xml_path"]],
    on=["file_id", "roi_index"],
    how="left",
    validate="one_to_one",
)
mass_manifest["sample_id"] = mass_manifest.apply(lambda r: f"{int(r.file_id)}_mass{int(r.roi_index):02d}", axis=1)
mass_manifest["dicom_path"] = mass_manifest["dicom_path"].map(lambda p: str(resolve_project_path(p)))

print("Tissue samples:", len(tissue_manifest), tissue_manifest["label"].value_counts().to_dict())
print("Mass samples:", len(mass_manifest), mass_manifest["label"].value_counts().to_dict())
print("Unique tissue source DICOMs:", tissue_manifest["dicom_path"].nunique())
print("Unique mass source DICOMs:", mass_manifest["dicom_path"].nunique())

tissue_manifest.to_csv(OUT / "original_tissue_manifest.csv", index=False)
mass_manifest.to_csv(OUT / "original_mass_manifest.csv", index=False)
display(tissue_manifest.head(3))
display(mass_manifest.head(3))

Tissue samples: 434 {'fatty': 251, 'fibroglandular': 183}
Mass samples: 115 {'malignant': 74, 'benign': 41}
Unique tissue source DICOMs: 100
Unique mass source DICOMs: 106


,sample_id,file_id,label,class_id,acr,dicom_path,y0,y1,x0,x1,orig_size
0,20586908_fatty_000,20586908,fatty,0,2,/home/nabeel/project34/Project34/data/raw/inbr...,1135,1407,1804,2076,273
1,20586908_fatty_001,20586908,fatty,0,2,/home/nabeel/project34/Project34/data/raw/inbr...,1159,1509,2530,2880,351
2,20586908_fatty_002,20586908,fatty,0,2,/home/nabeel/project34/Project34/data/raw/inbr...,2149,2499,1915,2265,351


,file_id,roi_index,label,class_id,birads,patch_npy,y0,y1,x0,x1,dicom_path,xml_path,sample_id
0,20586908,0,benign,0,2,../data/outputs/final_mass_replication/mass_pa...,963,1117,2373,2511,/home/nabeel/project34/Project34/data/raw/inbr...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,20586908_mass00
1,20586908,1,benign,0,2,../data/outputs/final_mass_replication/mass_pa...,988,1221,3091,3327,/home/nabeel/project34/Project34/data/raw/inbr...,../data/raw/kaggle_inbreast/AllXML/20586908.xml,20586908_mass01
2,20586934,0,malignant,1,5,../data/outputs/final_mass_replication/mass_pa...,2021,2222,105,331,/home/nabeel/project34/Project34/data/raw/inbr...,../data/raw/kaggle_inbreast/AllXML/20586934.xml,20586934_mass00


In [3]:
def build_patch_array(manifest, task):
    cache_path = OUT / f"{task}_original_patches_224.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        print(f"Loaded cached {task} patches:", arr.shape)
        return arr

    patches = []
    for i, row in enumerate(manifest.itertuples(index=False), start=1):
        inclusive = task == "tissue"  # Step 1.3 stores y1/x1 as inclusive endpoints.
        patch = crop_resize_raw(row.dicom_path, row.y0, row.y1, row.x0, row.x1, inclusive_y1x1=inclusive)
        patches.append(patch)
        if i % 100 == 0 or i == len(manifest):
            print(f"{task}: extracted {i}/{len(manifest)}")
    arr = np.stack(patches).astype(np.float32)
    np.save(cache_path, arr)
    print(f"Saved {task} patches:", cache_path, arr.shape)
    return arr


X_tissue_img = build_patch_array(tissue_manifest, "tissue")
y_tissue = tissue_manifest["class_id"].to_numpy(np.int64)
groups_tissue = tissue_manifest["file_id"].astype(str).to_numpy()

X_mass_img = build_patch_array(mass_manifest, "mass")
y_mass = mass_manifest["class_id"].to_numpy(np.int64)
groups_mass = mass_manifest["file_id"].astype(str).to_numpy()

print("Tissue patch intensity summary:", pd.Series(X_tissue_img.reshape(len(X_tissue_img), -1).mean(axis=1)).describe().round(4).to_dict())
print("Mass patch intensity summary:", pd.Series(X_mass_img.reshape(len(X_mass_img), -1).mean(axis=1)).describe().round(4).to_dict())

Loaded cached tissue patches: (434, 224, 224)
Loaded cached mass patches: (115, 224, 224)
Tissue patch intensity summary: {'count': 434.0, 'mean': 0.5553, 'std': 0.1208, 'min': 0.2311, '25%': 0.4683, '50%': 0.55, '75%': 0.6374, 'max': 0.8391}
Mass patch intensity summary: {'count': 115.0, 'mean': 0.6219, 'std': 0.1025, 'min': 0.3944, '25%': 0.5459, '50%': 0.638, '75%': 0.6984, 'max': 0.8447}


## 2. Shared Paper-Style Evaluation Helpers

The helper functions below mirror the current paper-style notebooks: averaged scattering features feed a subspace kNN ensemble; CNN features are extracted from a fine-tuned ResNet18 and then also evaluated with the same kNN ensemble; fusion concatenates standardised WS and CNN feature blocks.

In [4]:
def scattering_features(images, task, *, J=6, L=5, max_order=2):
    feat_path = OUT / f"{task}_ws_avg_features.npy"
    if feat_path.exists():
        feats = np.load(feat_path)
        print(f"Loaded cached {task} WS features:", feats.shape)
        return feats

    scattering = Scattering2D(J=J, shape=(IMG_SIZE, IMG_SIZE), L=L, max_order=max_order)
    features = []
    for i, img in enumerate(images, start=1):
        Sx = scattering(img)
        features.append(Sx.reshape(Sx.shape[0], -1).mean(axis=1))
        if i % 50 == 0 or i == len(images):
            print(f"{task}: WS {i}/{len(images)}")
    feats = np.asarray(features, dtype=np.float32)
    np.save(feat_path, feats)
    print(f"Saved {task} WS features:", feat_path, feats.shape)
    return feats


def mass_group_stratification_labels(y, groups):
    group_df = pd.DataFrame({"group": groups, "y": y}).drop_duplicates()
    group_labels = group_df.groupby("group")["y"].max().to_dict()
    return np.array([group_labels[g] for g in groups])


def split_indices(y, groups, seed, task):
    groups = np.asarray(groups)
    unique_groups = np.array(sorted(pd.unique(groups)))
    if task == "tissue":
        group_acr = tissue_manifest.drop_duplicates("file_id").set_index("file_id")["acr"].astype(int).to_dict()
        strat = np.array([group_acr[str(g)] for g in unique_groups])
    else:
        group_label = pd.DataFrame({"group": groups, "y": y}).drop_duplicates().groupby("group")["y"].max().to_dict()
        strat = np.array([group_label[str(g)] for g in unique_groups])

    train_groups, test_groups = train_test_split(
        unique_groups,
        test_size=0.20,
        random_state=seed,
        stratify=strat,
    )
    train_idx = np.where(np.isin(groups, train_groups))[0]
    test_idx = np.where(np.isin(groups, test_groups))[0]
    return train_idx, test_idx


def fit_transform_blocks(X_train, X_test, *, pca_var=None):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    if pca_var is not None:
        pca = PCA(n_components=pca_var, svd_solver="full", random_state=34)
        X_train_s = pca.fit_transform(X_train_s)
        X_test_s = pca.transform(X_test_s)
    return X_train_s, X_test_s


def evaluate_feature_set(X, y, groups, task, seed, *, pca_var=None, return_model=False):
    train_idx, test_idx = split_indices(y, groups, seed, task)
    X_train, X_test = fit_transform_blocks(X[train_idx], X[test_idx], pca_var=pca_var)
    y_train, y_test = y[train_idx], y[test_idx]
    clf = SubspaceKNN(random_state=seed)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    prob = clf.predict_proba(X_test)[:, 1]
    out = {
        "seed": seed,
        "test_n": len(test_idx),
        "acc": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "auroc": roc_auc_score(y_test, prob),
        "tn": confusion_matrix(y_test, pred, labels=[0, 1])[0, 0],
        "fp": confusion_matrix(y_test, pred, labels=[0, 1])[0, 1],
        "fn": confusion_matrix(y_test, pred, labels=[0, 1])[1, 0],
        "tp": confusion_matrix(y_test, pred, labels=[0, 1])[1, 1],
    }
    if return_model:
        return out, train_idx, test_idx
    return out


IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]


class PatchDataset(Dataset):
    def __init__(self, images, labels=None):
        self.images = images.astype(np.float32)
        self.labels = None if labels is None else labels.astype(np.int64)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx][None, :, :]).repeat(3, 1, 1)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        if self.labels is None:
            return x
        return x, torch.tensor(self.labels[idx], dtype=torch.long)


def make_resnet18_head(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def train_cnn_and_extract_features(images, y, groups, task, seed, *, epochs=60, batch_size=80, lr=1e-3, weight_decay=1e-4):
    set_seed(seed)
    feat_path = OUT / f"{task}_cnn_features_seed{seed}.npy"
    if feat_path.exists():
        feats = np.load(feat_path)
        print(f"Loaded cached {task} CNN features seed {seed}:", feats.shape)
        return feats

    train_idx, _ = split_indices(y, groups, seed, task)
    train_ds = PatchDataset(images[train_idx], y[train_idx])
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=generator, num_workers=0)

    model = make_resnet18_head().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            losses.append(float(loss.detach().cpu()))
        print(f"{task} seed {seed} epoch {epoch + 1:02d}/{epochs} loss={np.mean(losses):.4f}")

    extractor = nn.Sequential(*(list(model.children())[:-1])).to(DEVICE)
    extractor.eval()
    all_loader = DataLoader(PatchDataset(images), batch_size=batch_size, shuffle=False, num_workers=0)
    feats = []
    with torch.no_grad():
        for xb in all_loader:
            xb = xb.to(DEVICE)
            z = extractor(xb).flatten(1).cpu().numpy()
            feats.append(z)
    feats = np.vstack(feats).astype(np.float32)
    np.save(feat_path, feats)
    print(f"Saved {task} CNN features seed {seed}:", feat_path, feats.shape)
    return feats


def cv_score_for_features(X, y, groups, seed, task, *, pca_var=None):
    splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
    scores = []
    for tr, va in splitter.split(X, y, groups):
        X_tr, X_va = fit_transform_blocks(X[tr], X[va], pca_var=pca_var)
        clf = SubspaceKNN(random_state=seed)
        clf.fit(X_tr, y[tr])
        scores.append(accuracy_score(y[va], clf.predict(X_va)))
    return float(np.mean(scores)), float(np.std(scores, ddof=1))


def cv_score_for_fusion(ws_features, cnn_features, y, groups, seed):
    splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
    scores = []
    for tr, va in splitter.split(ws_features, y, groups):
        ws_tr, ws_va = fit_transform_blocks(ws_features[tr], ws_features[va])
        cnn_tr, cnn_va = fit_transform_blocks(cnn_features[tr], cnn_features[va])
        X_tr = np.hstack([ws_tr, cnn_tr])
        X_va = np.hstack([ws_va, cnn_va])
        clf = SubspaceKNN(random_state=seed)
        clf.fit(X_tr, y[tr])
        scores.append(accuracy_score(y[va], clf.predict(X_va)))
    return float(np.mean(scores)), float(np.std(scores, ddof=1))


def summarize_rows(perseed, role, method):
    rep = perseed[(perseed["role"] == role) & (perseed["method"] == method)].copy()
    cv = rep["cv_acc"].dropna()
    return {
        "Method": method,
        "Role": role,
        "AUROC": f"{rep['auroc'].mean():.3f} ± {rep['auroc'].std(ddof=1):.3f}",
        "Test acc": f"{rep['acc'].mean():.3f} ± {rep['acc'].std(ddof=1):.3f}",
        "CV (seed 34)": "" if cv.empty else f"{cv.iloc[0]:.3f} ± {rep['cv_sd'].dropna().iloc[0]:.3f}",
        "F1": f"{rep['f1'].mean():.3f}",
        "AUROC_num": rep["auroc"].mean(),
        "acc_num": rep["acc"].mean(),
        "auroc_sd": rep["auroc"].std(ddof=1),
        "acc_sd": rep["acc"].std(ddof=1),
    }


def run_task(task, images, y, groups, ws_features, *, cnn_epochs=60):
    perseed_path = OUT / f"perseed_{task}_original.csv"
    completed = pd.read_csv(perseed_path) if perseed_path.exists() else pd.DataFrame()
    rows = [] if completed.empty else completed.to_dict("records")

    done = set()
    if not completed.empty:
        done = set(zip(completed["seed"], completed["role"], completed["method"]))

    for seed in SEEDS:
        seed_rows = []
        if (seed, "Original-pixel investigation", "WS-only (averaged)") not in done:
            out = evaluate_feature_set(ws_features, y, groups, task, seed)
            out.update({"role": "Original-pixel investigation", "method": "WS-only (averaged)", "cv_acc": np.nan, "cv_sd": np.nan})
            seed_rows.append(out)

        cnn_features = train_cnn_and_extract_features(images, y, groups, task, seed, epochs=cnn_epochs)
        if (seed, "Original-pixel investigation", "CNN-only (ResNet18→kNN)") not in done:
            out = evaluate_feature_set(cnn_features, y, groups, task, seed)
            out.update({"role": "Original-pixel investigation", "method": "CNN-only (ResNet18→kNN)", "cv_acc": np.nan, "cv_sd": np.nan})
            seed_rows.append(out)

        train_idx, test_idx = split_indices(y, groups, seed, task)
        ws_train, ws_test = fit_transform_blocks(ws_features[train_idx], ws_features[test_idx])
        cnn_train, cnn_test = fit_transform_blocks(cnn_features[train_idx], cnn_features[test_idx])
        fusion = np.zeros((len(y), ws_train.shape[1] + cnn_train.shape[1]), dtype=np.float32)
        fusion[train_idx] = np.hstack([ws_train, cnn_train])
        fusion[test_idx] = np.hstack([ws_test, cnn_test])
        if (seed, "Original-pixel investigation", "CNN+WS fusion (concat)") not in done:
            clf = SubspaceKNN(random_state=seed)
            clf.fit(fusion[train_idx], y[train_idx])
            pred = clf.predict(fusion[test_idx])
            prob = clf.predict_proba(fusion[test_idx])[:, 1]
            cm = confusion_matrix(y[test_idx], pred, labels=[0, 1])
            out = {
                "seed": seed,
                "test_n": len(test_idx),
                "acc": accuracy_score(y[test_idx], pred),
                "precision": precision_score(y[test_idx], pred, zero_division=0),
                "recall": recall_score(y[test_idx], pred, zero_division=0),
                "f1": f1_score(y[test_idx], pred, zero_division=0),
                "auroc": roc_auc_score(y[test_idx], prob),
                "tn": cm[0, 0], "fp": cm[0, 1], "fn": cm[1, 0], "tp": cm[1, 1],
                "role": "Original-pixel investigation",
                "method": "CNN+WS fusion (concat)",
                "cv_acc": np.nan,
                "cv_sd": np.nan,
            }
            seed_rows.append(out)

        if seed == 34:
            for row in seed_rows:
                if row["method"] == "WS-only (averaged)":
                    row["cv_acc"], row["cv_sd"] = cv_score_for_features(ws_features, y, groups, seed, task)
                elif row["method"] == "CNN-only (ResNet18→kNN)":
                    row["cv_acc"], row["cv_sd"] = cv_score_for_features(cnn_features, y, groups, seed, task)
                elif row["method"] == "CNN+WS fusion (concat)":
                    row["cv_acc"], row["cv_sd"] = cv_score_for_fusion(ws_features, cnn_features, y, groups, seed)

        rows.extend(seed_rows)
        pd.DataFrame(rows).to_csv(perseed_path, index=False)
        print(f"Saved {task} per-seed results after seed {seed}: {perseed_path}")

    perseed = pd.DataFrame(rows).sort_values(["method", "seed"])
    master = pd.DataFrame([
        summarize_rows(perseed, "Original-pixel investigation", "WS-only (averaged)"),
        summarize_rows(perseed, "Original-pixel investigation", "CNN-only (ResNet18→kNN)"),
        summarize_rows(perseed, "Original-pixel investigation", "CNN+WS fusion (concat)"),
    ])
    master.to_csv(OUT / f"master_table_{task}_original.csv", index=False)
    return perseed, master


X_tissue_ws = scattering_features(X_tissue_img, "tissue")
X_mass_ws = scattering_features(X_mass_img, "mass")

Loaded cached tissue WS features: (434, 406)
Loaded cached mass WS features: (115, 406)


## 3. Run Original-Pixel Paper-Style Pipeline

This is the expensive part: it fine-tunes ResNet18 once per seed for tissue and mass, using the same seed list and paper-style hyperparameters as the main replication notebooks. Results are cached after each seed.

In [5]:
tissue_perseed, tissue_master = run_task("tissue", X_tissue_img, y_tissue, groups_tissue, X_tissue_ws)
mass_perseed, mass_master = run_task("mass", X_mass_img, y_mass, groups_mass, X_mass_ws)

print("Original-pixel tissue results")
display(tissue_master)
print("Original-pixel mass results")
display(mass_master)

Loaded cached tissue CNN features seed 1: (434, 512)
Saved tissue per-seed results after seed 1: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation/perseed_tissue_original.csv
Loaded cached tissue CNN features seed 7: (434, 512)
Saved tissue per-seed results after seed 7: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation/perseed_tissue_original.csv
Loaded cached tissue CNN features seed 34: (434, 512)
Saved tissue per-seed results after seed 34: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation/perseed_tissue_original.csv
Loaded cached tissue CNN features seed 42: (434, 512)
Saved tissue per-seed results after seed 42: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation/perseed_tissue_original.csv
Loaded cached tissue CNN features seed 99: (434, 512)
Saved tissue per-seed results after seed 99: /home/nabeel/project34/Project34/data/outputs/original_image_patch_invest

Saved mass per-seed results after seed 34: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation/perseed_mass_original.csv
Loaded cached mass CNN features seed 42: (115, 512)
Saved mass per-seed results after seed 42: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation/perseed_mass_original.csv
Loaded cached mass CNN features seed 99: (115, 512)
Saved mass per-seed results after seed 99: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation/perseed_mass_original.csv
Original-pixel tissue results


,Method,Role,AUROC,Test acc,CV (seed 34),F1,AUROC_num,acc_num,auroc_sd,acc_sd
0,WS-only (averaged),Original-pixel investigation,0.812 ± 0.074,0.767 ± 0.059,0.785 ± 0.079,0.736,0.811597,0.766792,0.073832,0.059449
1,CNN-only (ResNet18→kNN),Original-pixel investigation,0.841 ± 0.076,0.840 ± 0.069,0.975 ± 0.020,0.798,0.840503,0.839840,0.075868,0.068644
2,CNN+WS fusion (concat),Original-pixel investigation,0.868 ± 0.079,0.854 ± 0.065,0.975 ± 0.026,0.815,0.867758,0.853565,0.078759,0.064873


Original-pixel mass results


,Method,Role,AUROC,Test acc,CV (seed 34),F1,AUROC_num,acc_num,auroc_sd,acc_sd
0,WS-only (averaged),Original-pixel investigation,0.797 ± 0.021,0.700 ± 0.073,0.721 ± 0.137,0.754,0.797356,0.699732,0.020827,0.072754
1,CNN-only (ResNet18→kNN),Original-pixel investigation,0.644 ± 0.047,0.684 ± 0.060,0.853 ± 0.079,0.757,0.643620,0.683589,0.047263,0.059786
2,CNN+WS fusion (concat),Original-pixel investigation,0.722 ± 0.035,0.733 ± 0.042,0.845 ± 0.109,0.797,0.722180,0.733489,0.035422,0.042493


## 4. Compare Against Current Paper-Style Pipeline

The comparison below uses the current outputs from `final_tissue_replication` and `final_mass_replication`. This makes the investigation directly comparable to the current paper-style pipeline without touching the existing notebooks.

In [6]:
def canonical_method(s):
    return str(s).replace("→", "->")


def compare_with_current(task, original_master):
    current_path = DATA / "outputs" / ("final_tissue_replication" if task == "tissue" else "final_mass_replication") / f"master_table_{task}.csv"
    current = pd.read_csv(current_path)
    current = current[current["Role"] == "Replication"].copy()
    current["method_key"] = current["Method"].map(canonical_method)
    original = original_master.copy()
    original["method_key"] = original["Method"].map(canonical_method)
    cmp = current.merge(
        original,
        on="method_key",
        suffixes=("_current_preprocessed", "_original_pixels"),
        how="inner",
    )
    cmp["task"] = task
    cmp["AUROC_delta_original_minus_current"] = cmp["AUROC_num_original_pixels"] - cmp["AUROC_num_current_preprocessed"]
    cmp["acc_delta_original_minus_current"] = cmp["acc_num_original_pixels"] - cmp["acc_num_current_preprocessed"]
    cols = [
        "task",
        "Method_current_preprocessed",
        "AUROC_current_preprocessed",
        "AUROC_original_pixels",
        "AUROC_delta_original_minus_current",
        "Test acc_current_preprocessed",
        "Test acc_original_pixels",
        "acc_delta_original_minus_current",
        "CV (seed 34)_current_preprocessed",
        "CV (seed 34)_original_pixels",
    ]
    cmp = cmp[cols]
    cmp.to_csv(OUT / f"comparison_{task}_original_vs_current.csv", index=False)
    return cmp


tissue_cmp = compare_with_current("tissue", tissue_master)
mass_cmp = compare_with_current("mass", mass_master)
combined_cmp = pd.concat([tissue_cmp, mass_cmp], ignore_index=True)
combined_cmp.to_csv(OUT / "comparison_original_vs_current.csv", index=False)

pd.set_option("display.max_colwidth", 80)
print("Tissue: original pixels minus current preprocessed")
display(tissue_cmp)
print("Mass: original pixels minus current preprocessed")
display(mass_cmp)
print("Saved combined comparison:", OUT / "comparison_original_vs_current.csv")

Tissue: original pixels minus current preprocessed


,task,Method_current_preprocessed,AUROC_current_preprocessed,AUROC_original_pixels,AUROC_delta_original_minus_current,Test acc_current_preprocessed,Test acc_original_pixels,acc_delta_original_minus_current,CV (seed 34)_current_preprocessed,CV (seed 34)_original_pixels
0,tissue,WS-only (averaged),0.895 ± 0.070,0.812 ± 0.074,-0.082913,0.839 ± 0.071,0.767 ± 0.059,-0.071856,0.812 ± 0.073,0.785 ± 0.079
1,tissue,CNN-only (ResNet18→kNN),0.979 ± 0.010,0.841 ± 0.076,-0.138905,0.979 ± 0.010,0.840 ± 0.069,-0.139566,1.000 ± 0.000 (opt.),0.975 ± 0.020
2,tissue,CNN+WS fusion (concat),0.988 ± 0.007,0.868 ± 0.079,-0.120203,0.964 ± 0.030,0.854 ± 0.065,-0.110190,1.000 ± 0.000 (opt.),0.975 ± 0.026


Mass: original pixels minus current preprocessed


,task,Method_current_preprocessed,AUROC_current_preprocessed,AUROC_original_pixels,AUROC_delta_original_minus_current,Test acc_current_preprocessed,Test acc_original_pixels,acc_delta_original_minus_current,CV (seed 34)_current_preprocessed,CV (seed 34)_original_pixels
0,mass,WS-only (averaged),0.726 ± 0.030,0.797 ± 0.021,0.071126,0.701 ± 0.075,0.700 ± 0.073,-0.001194,0.711 ± 0.113,0.721 ± 0.137
1,mass,CNN-only (ResNet18→kNN),0.762 ± 0.117,0.644 ± 0.047,-0.118746,0.766 ± 0.067,0.684 ± 0.060,-0.082354,0.989 ± 0.033 (opt.),0.853 ± 0.079
2,mass,CNN+WS fusion (concat),0.816 ± 0.057,0.722 ± 0.035,-0.094009,0.792 ± 0.058,0.733 ± 0.042,-0.058295,0.978 ± 0.044 (opt.),0.845 ± 0.109


Saved combined comparison: /home/nabeel/project34/Project34/data/outputs/original_image_patch_investigation/comparison_original_vs_current.csv


## 5. Pixel-Source Sensitivity for WS-Only

This section tests the supervisor's idea more finely by creating a small ladder between the current fully preprocessed patches and the original-pixel patches.

The aim is not to replace the main replication. It is to ask which pixel source moves the WS-only results closer to Razali's reported values.

Variants:

- `current_full_preprocessed`: the existing Step 2.1/2.2 paper-style replication outputs.
- `raw_minmax`: original DICOM pixels with only full-image min-max scaling; already evaluated above.
- `raw_percentile`: original DICOM pixels with only 1st/99th percentile scaling, matching the normalisation step but not masking/CLAHE.
- `masked_no_clahe`: percentile-scaled DICOM image with the existing breast and pectoral masks applied, but no CLAHE.

Only WS-only is tested here because it directly isolates the effect of pixel source and is inexpensive enough to run across all five protocol seeds.

In [7]:
PAPER_TARGETS = {
    ("tissue", "WS-only (averaged)"): {"auroc": 0.94, "acc": 0.914},
    ("tissue", "CNN-only (ResNet18→kNN)"): {"auroc": 0.90, "acc": 0.774},
    ("tissue", "CNN+WS fusion (concat)"): {"auroc": 0.96, "acc": 0.935},
    ("mass", "WS-only (averaged)"): {"auroc": 0.80, "acc": 0.833},
    ("mass", "CNN-only (ResNet18→kNN)"): {"auroc": 0.80, "acc": 0.833},
    ("mass", "CNN+WS fusion (concat)"): {"auroc": 1.00, "acc": 0.917},
}


def load_dicom_percentile(path, lo_pct=1, hi_pct=99):
    ds = pydicom.dcmread(str(resolve_project_path(path)))
    img = ds.pixel_array.astype(np.float32)
    lo, hi = np.percentile(img, [lo_pct, hi_pct])
    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)
    return np.clip((img - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)


_percentile_cache = {}
_mask_cache = {}

def cached_percentile_image(path):
    path = resolve_project_path(path)
    key = str(path)
    if key not in _percentile_cache:
        _percentile_cache[key] = load_dicom_percentile(path)
    return _percentile_cache[key]


def cached_mask(path):
    path = resolve_project_path(path)
    key = str(path)
    if key not in _mask_cache:
        _mask_cache[key] = np.load(path).astype(bool)
    return _mask_cache[key]


preproc_masks = preproc.copy()
preproc_masks["dicom_resolved"] = preproc_masks["dicom"].map(lambda p: str(resolve_project_path(p)))
preproc_masks["breast_resolved"] = preproc_masks["breast_mask_npy"].map(lambda p: str(resolve_project_path(p)))
preproc_masks["pect_resolved"] = preproc_masks["pect_mask_npy"].map(lambda p: str(resolve_project_path(p)))
dicom_to_masks = preproc_masks.set_index("dicom_resolved")[["breast_resolved", "pect_resolved"]].to_dict("index")


def source_image_for_variant(dicom_path, variant):
    dicom_path = str(resolve_project_path(dicom_path))
    if variant == "raw_minmax":
        return cached_raw_image(dicom_path)
    if variant == "raw_percentile":
        return cached_percentile_image(dicom_path)
    if variant == "masked_no_clahe":
        img = cached_percentile_image(dicom_path).copy()
        masks = dicom_to_masks[dicom_path]
        breast = cached_mask(masks["breast_resolved"])
        pect = cached_mask(masks["pect_resolved"])
        img[~breast] = 0.0
        img[pect] = 0.0
        return img
    raise ValueError(f"Unknown pixel-source variant: {variant}")


def crop_resize_from_image(img, y0, y1, x0, x1, *, inclusive_y1x1=False):
    h, w = img.shape
    y0 = int(max(0, min(h, round(y0))))
    x0 = int(max(0, min(w, round(x0))))
    y1 = int(round(y1)) + (1 if inclusive_y1x1 else 0)
    x1 = int(round(x1)) + (1 if inclusive_y1x1 else 0)
    y1 = int(max(y0 + 1, min(h, y1)))
    x1 = int(max(x0 + 1, min(w, x1)))
    crop = img[y0:y1, x0:x1]
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)


def build_pixel_variant_array(manifest, task, variant):
    cache_path = OUT / f"{task}_{variant}_patches_224.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        print(f"Loaded cached {task} {variant} patches:", arr.shape)
        return arr
    patches = []
    for i, row in enumerate(manifest.itertuples(index=False), start=1):
        img = source_image_for_variant(row.dicom_path, variant)
        inclusive = task == "tissue"
        patches.append(crop_resize_from_image(img, row.y0, row.y1, row.x0, row.x1, inclusive_y1x1=inclusive))
        if i % 100 == 0 or i == len(manifest):
            print(f"{task} {variant}: extracted {i}/{len(manifest)}")
    arr = np.stack(patches).astype(np.float32)
    np.save(cache_path, arr)
    print(f"Saved {task} {variant} patches:", cache_path, arr.shape)
    return arr


def scattering_features_variant(images, task, variant):
    feat_path = OUT / f"{task}_{variant}_ws_avg_features.npy"
    if feat_path.exists():
        feats = np.load(feat_path)
        print(f"Loaded cached {task} {variant} WS features:", feats.shape)
        return feats
    scattering = Scattering2D(J=6, shape=(IMG_SIZE, IMG_SIZE), L=5, max_order=2)
    features = []
    for i, img in enumerate(images, start=1):
        Sx = scattering(img)
        features.append(Sx.reshape(Sx.shape[0], -1).mean(axis=1))
        if i % 50 == 0 or i == len(images):
            print(f"{task} {variant}: WS {i}/{len(images)}")
    feats = np.asarray(features, dtype=np.float32)
    np.save(feat_path, feats)
    print(f"Saved {task} {variant} WS features:", feat_path, feats.shape)
    return feats


def run_ws_pixel_variant(task, manifest, y, groups, variant):
    if variant == "raw_minmax":
        images = X_tissue_img if task == "tissue" else X_mass_img
        features = X_tissue_ws if task == "tissue" else X_mass_ws
    else:
        images = build_pixel_variant_array(manifest, task, variant)
        features = scattering_features_variant(images, task, variant)

    rows = []
    for seed in SEEDS:
        out = evaluate_feature_set(features, y, groups, task, seed)
        out.update({"task": task, "variant": variant, "method": "WS-only (averaged)", "cv_acc": np.nan, "cv_sd": np.nan})
        if seed == 34:
            out["cv_acc"], out["cv_sd"] = cv_score_for_features(features, y, groups, seed, task)
        rows.append(out)
    return pd.DataFrame(rows)


def summarize_ws_variant(perseed):
    rows = []
    for (task, variant), d in perseed.groupby(["task", "variant"]):
        target = PAPER_TARGETS[(task, "WS-only (averaged)")]
        cv = d["cv_acc"].dropna()
        cv_sd = d["cv_sd"].dropna()
        rows.append({
            "task": task,
            "variant": variant,
            "method": "WS-only (averaged)",
            "auroc": d["auroc"].mean(),
            "auroc_sd": d["auroc"].std(ddof=1),
            "test_acc": d["acc"].mean(),
            "test_acc_sd": d["acc"].std(ddof=1),
            "cv_acc_seed34": np.nan if cv.empty else cv.iloc[0],
            "cv_sd_seed34": np.nan if cv_sd.empty else cv_sd.iloc[0],
            "paper_auroc": target["auroc"],
            "paper_test_acc": target["acc"],
            "auroc_abs_gap_to_paper": abs(d["auroc"].mean() - target["auroc"]),
            "test_acc_abs_gap_to_paper": abs(d["acc"].mean() - target["acc"]),
        })
    return pd.DataFrame(rows).sort_values(["task", "auroc_abs_gap_to_paper", "test_acc_abs_gap_to_paper"])


variant_frames = []
for task, manifest, y, groups in [
    ("tissue", tissue_manifest, y_tissue, groups_tissue),
    ("mass", mass_manifest, y_mass, groups_mass),
]:
    for variant in ["raw_minmax", "raw_percentile", "masked_no_clahe"]:
        variant_frames.append(run_ws_pixel_variant(task, manifest, y, groups, variant))

ws_pixel_perseed = pd.concat(variant_frames, ignore_index=True)
ws_pixel_perseed.to_csv(OUT / "ws_pixel_source_variant_perseed.csv", index=False)
ws_pixel_summary = summarize_ws_variant(ws_pixel_perseed)

# Add the current full-preprocessing baseline from the main replication master tables.
current_rows = []
for task in ["tissue", "mass"]:
    current_path = DATA / "outputs" / ("final_tissue_replication" if task == "tissue" else "final_mass_replication") / f"master_table_{task}.csv"
    current = pd.read_csv(current_path)
    row = current[current["Method"] == "WS-only (averaged)"].iloc[0]
    target = PAPER_TARGETS[(task, "WS-only (averaged)")]
    current_rows.append({
        "task": task,
        "variant": "current_full_preprocessed",
        "method": "WS-only (averaged)",
        "auroc": row["AUROC_num"],
        "auroc_sd": row["auroc_sd"],
        "test_acc": row["acc_num"],
        "test_acc_sd": row["acc_sd"],
        "cv_acc_seed34": np.nan,
        "cv_sd_seed34": np.nan,
        "paper_auroc": target["auroc"],
        "paper_test_acc": target["acc"],
        "auroc_abs_gap_to_paper": abs(row["AUROC_num"] - target["auroc"]),
        "test_acc_abs_gap_to_paper": abs(row["acc_num"] - target["acc"]),
    })
ws_pixel_summary = pd.concat([pd.DataFrame(current_rows), ws_pixel_summary], ignore_index=True)
ws_pixel_summary = ws_pixel_summary.sort_values(["task", "auroc_abs_gap_to_paper", "test_acc_abs_gap_to_paper"])
ws_pixel_summary.to_csv(OUT / "ws_pixel_source_variant_summary.csv", index=False)

display(ws_pixel_summary)

Loaded cached tissue raw_percentile patches: (434, 224, 224)
Loaded cached tissue raw_percentile WS features: (434, 406)


Loaded cached tissue masked_no_clahe patches: (434, 224, 224)
Loaded cached tissue masked_no_clahe WS features: (434, 406)


Loaded cached mass raw_percentile patches: (115, 224, 224)
Loaded cached mass raw_percentile WS features: (115, 406)


Loaded cached mass masked_no_clahe patches: (115, 224, 224)
Loaded cached mass masked_no_clahe WS features: (115, 406)


,task,variant,method,auroc,auroc_sd,test_acc,test_acc_sd,cv_acc_seed34,cv_sd_seed34,paper_auroc,paper_test_acc,auroc_abs_gap_to_paper,test_acc_abs_gap_to_paper
2,mass,raw_minmax,WS-only (averaged),0.797356,0.020827,0.699732,0.072754,0.721212,0.136775,0.80,0.833,0.002644,0.133268
3,mass,masked_no_clahe,WS-only (averaged),0.805375,0.044808,0.741094,0.057112,0.746212,0.116723,0.80,0.833,0.005375,0.091906
4,mass,raw_percentile,WS-only (averaged),0.779093,0.045274,0.715977,0.072679,0.737121,0.118124,0.80,0.833,0.020907,0.117023
1,mass,current_full_preprocessed,WS-only (averaged),0.726230,0.029968,0.700926,0.075071,NaN,NaN,0.80,0.833,0.073770,0.132074
0,tissue,current_full_preprocessed,WS-only (averaged),0.894511,0.069980,0.838647,0.071351,NaN,NaN,0.94,0.914,0.045489,0.075353
5,tissue,masked_no_clahe,WS-only (averaged),0.881034,0.077330,0.838436,0.064954,0.813931,0.057941,0.94,0.914,0.058966,0.075564
6,tissue,raw_percentile,WS-only (averaged),0.881034,0.077330,0.838436,0.064954,0.813931,0.057941,0.94,0.914,0.058966,0.075564
7,tissue,raw_minmax,WS-only (averaged),0.811597,0.073832,0.766792,0.059449,0.784622,0.079295,0.94,0.914,0.128403,0.147208


## 6. CNN Branch Sensitivity Screen

The main paper-style notebooks already implement the CNN branch using the hyperparameters Razali et al. explicitly report. However, the paper does not fully specify several implementation choices, including fine-tuning policy, exact normalisation, and whether CNN predictions came directly from the network or from CNN features passed to the subspace kNN ensemble.

This section tests plausible interpretations of those missing details. It is a **seed-34 screen** rather than a full five-seed replacement, because each row requires CNN training and Razali's paper values are themselves single-split values.

All tests are kept inside this investigation notebook and use the current preprocessed patch set, not the original-pixel patches, because this section is about CNN branch details rather than pixel source.

In [8]:
def load_current_preprocessed_patches(task):
    cache_path = OUT / f"{task}_current_preprocessed_patches_224.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        print(f"Loaded cached {task} current preprocessed patches:", arr.shape)
        return arr
    manifest_path = DATA / "outputs" / ("final_tissue_replication" if task == "tissue" else "final_mass_replication") / f"{task}_manifest.csv"
    manifest = pd.read_csv(manifest_path)
    patches = []
    for i, path in enumerate(manifest["patch_npy"], start=1):
        patch = np.load(resolve_project_path(path)).astype(np.float32)
        if patch.shape != (IMG_SIZE, IMG_SIZE):
            patch = cv2.resize(patch, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)
        patches.append(patch)
        if i % 100 == 0 or i == len(manifest):
            print(f"{task}: loaded current patch {i}/{len(manifest)}")
    arr = np.stack(patches).astype(np.float32)
    np.save(cache_path, arr)
    print(f"Saved current preprocessed patch cache: {cache_path}", arr.shape)
    return arr


class CNNVariantDataset(Dataset):
    def __init__(self, images, labels=None, *, imagenet_norm=True):
        self.images = images.astype(np.float32)
        self.labels = None if labels is None else labels.astype(np.int64)
        self.imagenet_norm = imagenet_norm

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx][None, :, :]).repeat(3, 1, 1)
        if self.imagenet_norm:
            x = (x - IMAGENET_MEAN) / IMAGENET_STD
        if self.labels is None:
            return x
        return x, torch.tensor(self.labels[idx], dtype=torch.long)


def set_backbone_trainable(model, trainable):
    for name, param in model.named_parameters():
        param.requires_grad = trainable or name.startswith("fc.")


def metrics_from_predictions(y_true, pred, prob):
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        "acc": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "auroc": roc_auc_score(y_true, prob),
        "tn": cm[0, 0], "fp": cm[0, 1], "fn": cm[1, 0], "tp": cm[1, 1],
    }


def train_cnn_variant(task, images, y, groups, variant_name, *, seed=34, epochs=60, freeze_backbone=False, imagenet_norm=True):
    set_seed(seed)
    train_idx, test_idx = split_indices(y, groups, seed, task)
    train_ds = CNNVariantDataset(images[train_idx], y[train_idx], imagenet_norm=imagenet_norm)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=80, shuffle=True, generator=generator, num_workers=0)

    model = make_resnet18_head().to(DEVICE)
    set_backbone_trainable(model, not freeze_backbone)
    opt = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            losses.append(float(loss.detach().cpu()))
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch + 1 == epochs:
            print(f"{task} {variant_name} epoch {epoch + 1:02d}/{epochs} loss={np.mean(losses):.4f}")

    model.eval()
    test_loader = DataLoader(CNNVariantDataset(images[test_idx], y[test_idx], imagenet_norm=imagenet_norm), batch_size=80, shuffle=False, num_workers=0)
    direct_probs = []
    direct_preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            logits = model(xb.to(DEVICE))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            direct_probs.append(probs[:, 1])
            direct_preds.append(probs.argmax(axis=1))
    direct_prob = np.concatenate(direct_probs)
    direct_pred = np.concatenate(direct_preds)
    direct = metrics_from_predictions(y[test_idx], direct_pred, direct_prob)

    extractor = nn.Sequential(*(list(model.children())[:-1])).to(DEVICE)
    extractor.eval()
    all_loader = DataLoader(CNNVariantDataset(images, imagenet_norm=imagenet_norm), batch_size=80, shuffle=False, num_workers=0)
    features = []
    with torch.no_grad():
        for xb in all_loader:
            z = extractor(xb.to(DEVICE)).flatten(1).cpu().numpy()
            features.append(z)
    features = np.vstack(features).astype(np.float32)

    knn = evaluate_feature_set(features, y, groups, task, seed)
    return direct, knn


def result_already_done(results, task, variant_name):
    if results.empty:
        return False
    needed = {"direct_softmax", "cnn_features_knn"}
    done = set(results[(results["task"] == task) & (results["variant"] == variant_name)]["classifier"])
    return needed.issubset(done)


def append_cnn_variant_results(rows):
    path = OUT / "cnn_branch_variant_seed34_results.csv"
    old = pd.read_csv(path) if path.exists() else pd.DataFrame()
    new = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)
    new.to_csv(path, index=False)
    return new


CNN_VARIANTS = [
    {"variant": "faithful_full_finetune_60ep", "epochs": 60, "freeze_backbone": False, "imagenet_norm": True},
    {"variant": "freeze_backbone_60ep", "epochs": 60, "freeze_backbone": True, "imagenet_norm": True},
    {"variant": "short_finetune_20ep", "epochs": 20, "freeze_backbone": False, "imagenet_norm": True},
    {"variant": "no_imagenet_norm_60ep", "epochs": 60, "freeze_backbone": False, "imagenet_norm": False},
]

cnn_results_path = OUT / "cnn_branch_variant_seed34_results.csv"
cnn_results = pd.read_csv(cnn_results_path) if cnn_results_path.exists() else pd.DataFrame()

X_tissue_current = load_current_preprocessed_patches("tissue")
X_mass_current = load_current_preprocessed_patches("mass")

for task, images, y, groups in [
    ("tissue", X_tissue_current, y_tissue, groups_tissue),
    ("mass", X_mass_current, y_mass, groups_mass),
]:
    for cfg in CNN_VARIANTS:
        variant_name = cfg["variant"]
        if result_already_done(cnn_results, task, variant_name):
            print(f"Skipping completed CNN variant: {task} {variant_name}")
            continue
        direct, knn = train_cnn_variant(task, images, y, groups, variant_name, seed=34, epochs=cfg["epochs"], freeze_backbone=cfg["freeze_backbone"], imagenet_norm=cfg["imagenet_norm"])
        new_rows = []
        for classifier, metrics in [("direct_softmax", direct), ("cnn_features_knn", knn)]:
            target = PAPER_TARGETS[(task, "CNN-only (ResNet18→kNN)")]
            new_rows.append({
                "task": task,
                "seed": 34,
                "variant": variant_name,
                "classifier": classifier,
                "epochs": cfg["epochs"],
                "freeze_backbone": cfg["freeze_backbone"],
                "imagenet_norm": cfg["imagenet_norm"],
                **metrics,
                "paper_auroc": target["auroc"],
                "paper_test_acc": target["acc"],
                "auroc_abs_gap_to_paper": abs(metrics["auroc"] - target["auroc"]),
                "test_acc_abs_gap_to_paper": abs(metrics["acc"] - target["acc"]),
            })
        cnn_results = append_cnn_variant_results(new_rows)
        print(f"Saved CNN variant results: {task} {variant_name}")

cnn_results = pd.read_csv(cnn_results_path)
cnn_results = cnn_results.sort_values(["task", "classifier", "auroc_abs_gap_to_paper", "test_acc_abs_gap_to_paper"])
cnn_results.to_csv(OUT / "cnn_branch_variant_seed34_summary.csv", index=False)
display(cnn_results)

Loaded cached tissue current preprocessed patches: (434, 224, 224)
Loaded cached mass current preprocessed patches: (115, 224, 224)
Skipping completed CNN variant: tissue faithful_full_finetune_60ep
Skipping completed CNN variant: tissue freeze_backbone_60ep
Skipping completed CNN variant: tissue short_finetune_20ep
Skipping completed CNN variant: tissue no_imagenet_norm_60ep
Skipping completed CNN variant: mass faithful_full_finetune_60ep
Skipping completed CNN variant: mass freeze_backbone_60ep
Skipping completed CNN variant: mass short_finetune_20ep
Skipping completed CNN variant: mass no_imagenet_norm_60ep


,task,seed,variant,classifier,epochs,freeze_backbone,imagenet_norm,acc,precision,recall,...,auroc,tn,fp,fn,tp,paper_auroc,paper_test_acc,auroc_abs_gap_to_paper,test_acc_abs_gap_to_paper,test_n
11,mass,34,freeze_backbone_60ep,cnn_features_knn,60,True,True,0.640000,0.700000,0.823529,...,0.683824,2,6,3,14,0.8,0.833,0.116176,0.193000,25.0
13,mass,34,short_finetune_20ep,cnn_features_knn,20,False,True,0.880000,0.937500,0.882353,...,0.930147,7,1,2,15,0.8,0.833,0.130147,0.047000,25.0
15,mass,34,no_imagenet_norm_60ep,cnn_features_knn,60,False,False,0.680000,0.846154,0.647059,...,0.661765,6,2,6,11,0.8,0.833,0.138235,0.153000,25.0
9,mass,34,faithful_full_finetune_60ep,cnn_features_knn,60,False,True,0.720000,0.750000,0.882353,...,0.573529,3,5,2,15,0.8,0.833,0.226471,0.113000,25.0
8,mass,34,faithful_full_finetune_60ep,direct_softmax,60,False,True,0.680000,0.736842,0.823529,...,0.794118,3,5,3,14,0.8,0.833,0.005882,0.153000,NaN
10,mass,34,freeze_backbone_60ep,direct_softmax,60,True,True,0.760000,0.761905,0.941176,...,0.742647,3,5,1,16,0.8,0.833,0.057353,0.073000,NaN
12,mass,34,short_finetune_20ep,direct_softmax,20,False,True,0.880000,0.888889,0.941176,...,0.948529,6,2,1,16,0.8,0.833,0.148529,0.047000,NaN
14,mass,34,no_imagenet_norm_60ep,direct_softmax,60,False,False,0.440000,0.666667,0.352941,...,0.514706,5,3,11,6,0.8,0.833,0.285294,0.393000,NaN
7,tissue,34,no_imagenet_norm_60ep,cnn_features_knn,60,False,False,0.955556,0.972222,0.921053,...,0.950911,51,1,3,35,0.9,0.774,0.050911,0.181556,90.0
3,tissue,34,freeze_backbone_60ep,cnn_features_knn,60,True,True,0.900000,0.939394,0.815789,...,0.973431,50,2,7,31,0.9,0.774,0.073431,0.126000,90.0


## 7. Closest-to-Razali Summary

These tables are a screening summary, not a replacement for the main replication. The preferred interpretation is: if a variant moves closer to Razali, it identifies an undocumented implementation detail that may explain the reproduction gap.

In [9]:
closest_ws_auroc = ws_pixel_summary.sort_values(["task", "auroc_abs_gap_to_paper"]).groupby("task", as_index=False).nth(0).reset_index(drop=True)
closest_ws_test = ws_pixel_summary.sort_values(["task", "test_acc_abs_gap_to_paper"]).groupby("task", as_index=False).nth(0).reset_index(drop=True)
closest_cnn_auroc = cnn_results.sort_values(["task", "classifier", "auroc_abs_gap_to_paper"]).groupby(["task", "classifier"], as_index=False).nth(0).reset_index(drop=True)
closest_cnn_test = cnn_results.sort_values(["task", "classifier", "test_acc_abs_gap_to_paper"]).groupby(["task", "classifier"], as_index=False).nth(0).reset_index(drop=True)

closest_ws_auroc.to_csv(OUT / "closest_ws_pixel_variant_by_auroc.csv", index=False)
closest_ws_test.to_csv(OUT / "closest_ws_pixel_variant_by_test_acc.csv", index=False)
closest_cnn_auroc.to_csv(OUT / "closest_cnn_variant_by_auroc.csv", index=False)
closest_cnn_test.to_csv(OUT / "closest_cnn_variant_by_test_acc.csv", index=False)

print("Closest WS pixel-source variant by AUROC gap to Razali")
display(closest_ws_auroc[["task", "variant", "auroc", "paper_auroc", "auroc_abs_gap_to_paper", "test_acc", "paper_test_acc", "test_acc_abs_gap_to_paper"]])

print("Closest WS pixel-source variant by test-accuracy gap to Razali")
display(closest_ws_test[["task", "variant", "test_acc", "paper_test_acc", "test_acc_abs_gap_to_paper", "auroc", "paper_auroc", "auroc_abs_gap_to_paper"]])

print("Closest CNN branch variant by AUROC gap to Razali")
display(closest_cnn_auroc[["task", "classifier", "variant", "auroc", "paper_auroc", "auroc_abs_gap_to_paper", "acc", "paper_test_acc", "test_acc_abs_gap_to_paper"]])

print("Closest CNN branch variant by test-accuracy gap to Razali")
display(closest_cnn_test[["task", "classifier", "variant", "acc", "paper_test_acc", "test_acc_abs_gap_to_paper", "auroc", "paper_auroc", "auroc_abs_gap_to_paper"]])

Closest WS pixel-source variant by AUROC gap to Razali


,task,variant,auroc,paper_auroc,auroc_abs_gap_to_paper,test_acc,paper_test_acc,test_acc_abs_gap_to_paper
0,mass,raw_minmax,0.797356,0.80,0.002644,0.699732,0.833,0.133268
1,tissue,current_full_preprocessed,0.894511,0.94,0.045489,0.838647,0.914,0.075353


Closest WS pixel-source variant by test-accuracy gap to Razali


,task,variant,test_acc,paper_test_acc,test_acc_abs_gap_to_paper,auroc,paper_auroc,auroc_abs_gap_to_paper
0,mass,masked_no_clahe,0.741094,0.833,0.091906,0.805375,0.80,0.005375
1,tissue,current_full_preprocessed,0.838647,0.914,0.075353,0.894511,0.94,0.045489


Closest CNN branch variant by AUROC gap to Razali


,task,classifier,variant,auroc,paper_auroc,auroc_abs_gap_to_paper,acc,paper_test_acc,test_acc_abs_gap_to_paper
0,mass,cnn_features_knn,freeze_backbone_60ep,0.683824,0.8,0.116176,0.640000,0.833,0.193000
1,mass,direct_softmax,faithful_full_finetune_60ep,0.794118,0.8,0.005882,0.680000,0.833,0.153000
2,tissue,cnn_features_knn,no_imagenet_norm_60ep,0.950911,0.9,0.050911,0.955556,0.774,0.181556
3,tissue,direct_softmax,freeze_backbone_60ep,0.988866,0.9,0.088866,0.966667,0.774,0.192667


Closest CNN branch variant by test-accuracy gap to Razali


,task,classifier,variant,acc,paper_test_acc,test_acc_abs_gap_to_paper,auroc,paper_auroc,auroc_abs_gap_to_paper
0,mass,cnn_features_knn,short_finetune_20ep,0.880000,0.833,0.047000,0.930147,0.8,0.130147
1,mass,direct_softmax,short_finetune_20ep,0.880000,0.833,0.047000,0.948529,0.8,0.148529
2,tissue,cnn_features_knn,freeze_backbone_60ep,0.900000,0.774,0.126000,0.973431,0.9,0.073431
3,tissue,direct_softmax,short_finetune_20ep,0.922222,0.774,0.148222,0.995445,0.9,0.095445


## 8. Region-Aware Extension with Masked/No-CLAHE Inputs

This section tests whether the best WS-only pixel-source insight (`masked_no_clahe`) also helps the region-aware mass extension.

It mirrors the Step 2.4 region-aware WS-map CNN-head setup inside this investigation notebook only:

- same 115 mass samples and image-level grouped splits,
- same XML mass polygon masks,
- same four regions: `full`, `inner`, `boundary`, `context`,
- same J=3/L=5 spatial WS tensors,
- same small CNN head and five-seed holdout protocol.

The only intended change is the image source used before region construction: percentile-scaled original DICOM image with breast/background and pectoral masks applied, but no CLAHE.

In [10]:
import plistlib
import re
import time
from skimage.draw import polygon
from torch.utils.data import TensorDataset

RA_REGIONS = ["full", "inner", "boundary", "context"]
RA_VARIANT = "+".join(RA_REGIONS)
RA_J = 3
RA_L = 5
RA_MAX_ORDER = 2
RA_PIXEL_SOURCE = "masked_no_clahe"

_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")


def _parse_point_any(p):
    if isinstance(p, (list, tuple)) and len(p) == 2:
        try:
            return float(p[0]), float(p[1])
        except Exception:
            return None
    if isinstance(p, str):
        nums = _num_re.findall(p)
        if len(nums) >= 2:
            return float(nums[0]), float(nums[1])
    return None


def _coerce_points(pts):
    out = []
    if pts is None:
        return np.zeros((0, 2), dtype=float)
    if isinstance(pts, str):
        nums = _num_re.findall(pts)
        for i in range(0, len(nums) - 1, 2):
            out.append((float(nums[i]), float(nums[i + 1])))
        return np.array(out, dtype=float) if out else np.zeros((0, 2), dtype=float)
    if isinstance(pts, (list, tuple)):
        for p in pts:
            xy = _parse_point_any(p)
            if xy is not None:
                out.append(xy)
            elif isinstance(p, (list, tuple)) and len(p) > 2:
                try:
                    flat = list(p)
                    for i in range(0, len(flat) - 1, 2):
                        out.append((float(flat[i]), float(flat[i + 1])))
                except Exception:
                    pass
    return np.array(out, dtype=float) if out else np.zeros((0, 2), dtype=float)


def load_rois_from_inbreast_xml(xml_path):
    with open(resolve_project_path(xml_path), "rb") as f:
        data = plistlib.load(f)
    img0 = data["Images"][0]
    out = []
    for roi in img0.get("ROIs", []):
        name = str(roi.get("Name", "")).strip()
        pts_raw = roi.get("Point_px", None)
        if pts_raw is None:
            pts_raw = roi.get("Points", None)
        pts = _coerce_points(pts_raw)
        if len(pts) >= 3:
            out.append({"name": name, "points": pts})
    return out


def polygon_to_mask(points, shape_hw):
    pts = np.asarray(points, dtype=float)
    if pts.shape[0] < 3:
        return np.zeros(shape_hw, dtype=bool)
    rr, cc = polygon(pts[:, 1], pts[:, 0], shape_hw)
    mask = np.zeros(shape_hw, dtype=bool)
    mask[rr, cc] = True
    return mask


def resize_mask_224(mask):
    return cv2.resize(mask.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST).astype(bool)


def resize_image_224(img):
    return cv2.resize(img.astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)


def make_ra_region_images(patch, roi_mask, erode_px=8, dilate_px=8):
    mask_u8 = roi_mask.astype(np.uint8)
    erode_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * erode_px + 1, 2 * erode_px + 1))
    dilate_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dilate_px + 1, 2 * dilate_px + 1))
    inner = cv2.erode(mask_u8, erode_kernel, iterations=1).astype(bool)
    dilated = cv2.dilate(mask_u8, dilate_kernel, iterations=1).astype(bool)
    boundary = dilated & (~inner)
    context = ~dilated
    return {
        "full": np.clip(patch, 0.0, 1.0).astype(np.float32),
        "inner": (patch * inner).astype(np.float32),
        "boundary": (patch * boundary).astype(np.float32),
        "context": (patch * context).astype(np.float32),
    }


def get_ra_roi_mask_for_row(row, image_shape):
    rois = [r for r in load_rois_from_inbreast_xml(row.xml_path) if r["name"].strip().lower() == "mass"]
    idx = int(row.roi_index)
    if idx >= len(rois):
        raise IndexError(f"ROI index {idx} not available in {row.xml_path}; found {len(rois)} mass ROIs")
    return polygon_to_mask(rois[idx]["points"], image_shape)


def build_masked_no_clahe_region_images(rebuild=False):
    cache_path = OUT / "region_aware_masked_no_clahe_region_images_224.npz"
    if cache_path.exists() and not rebuild:
        cached = np.load(cache_path, allow_pickle=True)
        return cached["region_images"], list(cached["region_names"])

    region_images = []
    skipped = []
    for i, row in enumerate(mass_manifest.itertuples(index=False), start=1):
        image = source_image_for_variant(row.dicom_path, RA_PIXEL_SOURCE)
        y0, y1, x0, x1 = int(row.y0), int(row.y1), int(row.x0), int(row.x1)
        try:
            full_mask = get_ra_roi_mask_for_row(row, image.shape)
            patch = resize_image_224(image[y0:y1, x0:x1])
            roi_mask = resize_mask_224(full_mask[y0:y1, x0:x1])
            if patch.max() <= 0 or roi_mask.sum() == 0:
                skipped.append((row.file_id, int(row.roi_index), "zero patch or empty mask"))
                continue
            regions = make_ra_region_images(patch, roi_mask)
            region_images.append(np.stack([regions[name] for name in RA_REGIONS], axis=0))
        except Exception as exc:
            skipped.append((row.file_id, int(row.roi_index), str(exc)))
        if i % 25 == 0 or i == len(mass_manifest):
            print(f"Region images {i}/{len(mass_manifest)}")

    if skipped:
        pd.DataFrame(skipped, columns=["file_id", "roi_index", "reason"]).to_csv(
            OUT / "region_aware_masked_no_clahe_skipped.csv", index=False
        )
        print("Skipped region-aware masked/no-CLAHE rows:", len(skipped))

    region_images = np.stack(region_images).astype(np.float32)
    np.savez_compressed(cache_path, region_images=region_images, region_names=np.array(RA_REGIONS))
    print("Saved masked/no-CLAHE region images:", cache_path, region_images.shape)
    return region_images, RA_REGIONS


def build_masked_no_clahe_ra_ws_tensors(rebuild=False):
    cache_path = OUT / f"region_aware_masked_no_clahe_ws_tensors_J{RA_J}_L{RA_L}_{RA_VARIANT.replace('+', '_')}.npz"
    if cache_path.exists() and not rebuild:
        cached = np.load(cache_path, allow_pickle=True)
        return cached["X"], cached["y"], cached["groups"], list(cached["region_names"])

    region_images, region_names = build_masked_no_clahe_region_images(rebuild=rebuild)
    scattering = Scattering2D(J=RA_J, shape=(IMG_SIZE, IMG_SIZE), L=RA_L, max_order=RA_MAX_ORDER)
    tensors = []
    for i, sample_regions in enumerate(region_images, start=1):
        region_tensors = []
        for region_img in sample_regions:
            S = np.asarray(scattering(region_img.astype(np.float32)), dtype=np.float32)
            if S.ndim == 4:
                S = S.reshape(-1, S.shape[-2], S.shape[-1])
            region_tensors.append(S)
        tensors.append(np.concatenate(region_tensors, axis=0))
        if i % 25 == 0 or i == len(region_images):
            print(f"Region-aware masked/no-CLAHE WS {i}/{len(region_images)}")

    X = np.stack(tensors).astype(np.float32)
    y = mass_manifest["class_id"].to_numpy(np.int64)
    groups = mass_manifest["file_id"].astype(str).to_numpy()
    np.savez_compressed(
        cache_path,
        X=X,
        y=y,
        groups=groups,
        region_names=np.array(region_names),
        J=np.array(RA_J),
        L=np.array(RA_L),
        max_order=np.array(RA_MAX_ORDER),
        pixel_source=np.array(RA_PIXEL_SOURCE),
    )
    print("Saved masked/no-CLAHE RA WS tensors:", cache_path, X.shape)
    return X, y, groups, region_names


class RAWSMapCNN(nn.Module):
    def __init__(self, channels, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.net(x)


def ra_stratified_group_holdout(groups, y, seed, test_size=0.2):
    group_df = pd.DataFrame({"group": groups, "y": y}).drop_duplicates("group")
    train_groups, test_groups = train_test_split(
        group_df["group"].to_numpy(),
        test_size=test_size,
        random_state=seed,
        stratify=group_df["y"].to_numpy(),
    )
    return np.isin(groups, train_groups), np.isin(groups, test_groups)


def ra_normalise_train_only(X, train_mask):
    mu = X[train_mask].mean(axis=(0, 2, 3), keepdims=True)
    sd = X[train_mask].std(axis=(0, 2, 3), keepdims=True) + 1e-6
    return ((X - mu) / sd).astype(np.float32)


def train_ra_one_seed(X, y, groups, seed, epochs=60, batch_size=24, lr=1e-3, weight_decay=3e-4, dropout=0.5):
    set_seed(seed)
    train_mask, test_mask = ra_stratified_group_holdout(groups, y, seed)
    Xn = ra_normalise_train_only(X, train_mask)

    X_train = torch.tensor(Xn[train_mask], dtype=torch.float32)
    y_train = torch.tensor(y[train_mask], dtype=torch.long)
    X_test = torch.tensor(Xn[test_mask], dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    model = RAWSMapCNN(X.shape[1], dropout=dropout).to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    start = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimiser.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimiser.step()

    model.eval()
    with torch.no_grad():
        train_prob = torch.softmax(model(X_train.to(DEVICE)).cpu(), dim=1)[:, 1].numpy()
        test_prob = torch.softmax(model(X_test.to(DEVICE)).cpu(), dim=1)[:, 1].numpy()

    y_train_np = y[train_mask]
    y_test_np = y[test_mask]
    train_pred = (train_prob >= 0.5).astype(int)
    test_pred = (test_prob >= 0.5).astype(int)

    return {
        "seed": seed,
        "variant": RA_VARIANT,
        "pixel_source": RA_PIXEL_SOURCE,
        "J": RA_J,
        "L": RA_L,
        "n_train": int(train_mask.sum()),
        "n_test": int(test_mask.sum()),
        "train_acc": accuracy_score(y_train_np, train_pred),
        "test_acc": accuracy_score(y_test_np, test_pred),
        "precision": precision_score(y_test_np, test_pred, zero_division=0),
        "recall": recall_score(y_test_np, test_pred, zero_division=0),
        "f1": f1_score(y_test_np, test_pred, zero_division=0),
        "auroc": roc_auc_score(y_test_np, test_prob) if len(np.unique(y_test_np)) == 2 else np.nan,
        "confusion_matrix": confusion_matrix(y_test_np, test_pred).tolist(),
        "seconds": time.time() - start,
    }


X_ra_masked, y_ra_masked, groups_ra_masked, region_names_ra_masked = build_masked_no_clahe_ra_ws_tensors(rebuild=False)
print("Masked/no-CLAHE region-aware tensor shape:", X_ra_masked.shape)
print("Class counts:", dict(zip(*np.unique(y_ra_masked, return_counts=True))))

ra_perseed_path = OUT / "region_aware_masked_no_clahe_ws_cnn_mass_per_seed.csv"
if ra_perseed_path.exists():
    ra_per_seed = pd.read_csv(ra_perseed_path)
    completed = set(ra_per_seed["seed"].astype(int))
else:
    ra_per_seed = pd.DataFrame()
    completed = set()

rows = [] if ra_per_seed.empty else ra_per_seed.to_dict("records")
for seed in SEEDS:
    if int(seed) in completed:
        print(f"Skipping completed masked/no-CLAHE region-aware seed {seed}")
        continue
    print(f"Running masked/no-CLAHE region-aware seed {seed}...")
    result = train_ra_one_seed(X_ra_masked, y_ra_masked, groups_ra_masked, seed)
    rows.append(result)
    pd.DataFrame(rows).to_csv(ra_perseed_path, index=False)
    print({k: result[k] for k in ["seed", "test_acc", "auroc", "precision", "recall", "f1"]})

ra_per_seed = pd.DataFrame(rows).sort_values("seed")
ra_per_seed.to_csv(ra_perseed_path, index=False)
metric_cols = ["test_acc", "auroc", "precision", "recall", "f1", "train_acc"]
ra_summary = ra_per_seed[metric_cols].agg(["mean", "std"]).T.reset_index()
ra_summary.columns = ["metric", "mean", "std"]
ra_summary.insert(0, "method", "Region-aware WS maps -> CNN head")
ra_summary.insert(1, "pixel_source", RA_PIXEL_SOURCE)
ra_summary.insert(2, "variant", RA_VARIANT)
ra_summary.insert(3, "J", RA_J)
ra_summary.insert(4, "L", RA_L)
ra_summary.insert(5, "n_seeds", len(SEEDS))
ra_summary.to_csv(OUT / "region_aware_masked_no_clahe_ws_cnn_mass_summary.csv", index=False)

current_ra_summary_path = DATA / "outputs" / "recovered_region_aware_ws_cnn" / "region_aware_ws_cnn_mass_summary.csv"
current_ra_summary = pd.read_csv(current_ra_summary_path)
current_pivot = current_ra_summary.pivot_table(index="metric", values=["mean", "std"], aggfunc="first")
masked_pivot = ra_summary.pivot_table(index="metric", values=["mean", "std"], aggfunc="first")
ra_comparison = pd.DataFrame({
    "metric": masked_pivot.index,
    "current_full_preprocessed_mean": current_pivot.loc[masked_pivot.index, "mean"].values,
    "masked_no_clahe_mean": masked_pivot["mean"].values,
    "delta_masked_minus_current": masked_pivot["mean"].values - current_pivot.loc[masked_pivot.index, "mean"].values,
    "current_full_preprocessed_std": current_pivot.loc[masked_pivot.index, "std"].values,
    "masked_no_clahe_std": masked_pivot["std"].values,
})
ra_comparison.to_csv(OUT / "region_aware_masked_no_clahe_vs_current_comparison.csv", index=False)

display(ra_per_seed[["seed", "n_train", "n_test", "train_acc", "test_acc", "auroc", "precision", "recall", "f1"]].round(3))
display(ra_summary.round(3))
display(ra_comparison.round(3))

Masked/no-CLAHE region-aware tensor shape: (115, 364, 28, 28)
Class counts: {np.int64(0): np.int64(41), np.int64(1): np.int64(74)}
Skipping completed masked/no-CLAHE region-aware seed 1
Skipping completed masked/no-CLAHE region-aware seed 7
Skipping completed masked/no-CLAHE region-aware seed 34
Skipping completed masked/no-CLAHE region-aware seed 42
Skipping completed masked/no-CLAHE region-aware seed 99


,seed,n_train,n_test,train_acc,test_acc,auroc,precision,recall,f1
0,1,89,26,1.0,0.923,0.950,0.938,0.938,0.938
1,7,93,22,1.0,0.909,0.971,1.000,0.867,0.929
2,34,90,25,1.0,0.920,0.971,0.941,0.941,0.941
3,42,92,23,1.0,0.957,1.000,0.941,1.000,0.970
4,99,91,24,1.0,0.917,0.961,0.938,0.938,0.938


,method,pixel_source,variant,J,L,n_seeds,metric,mean,std
0,Region-aware WS maps -> CNN head,masked_no_clahe,full+inner+boundary+context,3,5,5,test_acc,0.925,0.018
1,Region-aware WS maps -> CNN head,masked_no_clahe,full+inner+boundary+context,3,5,5,auroc,0.971,0.019
2,Region-aware WS maps -> CNN head,masked_no_clahe,full+inner+boundary+context,3,5,5,precision,0.951,0.027
3,Region-aware WS maps -> CNN head,masked_no_clahe,full+inner+boundary+context,3,5,5,recall,0.937,0.047
4,Region-aware WS maps -> CNN head,masked_no_clahe,full+inner+boundary+context,3,5,5,f1,0.943,0.016
5,Region-aware WS maps -> CNN head,masked_no_clahe,full+inner+boundary+context,3,5,5,train_acc,1.000,0.000


,metric,current_full_preprocessed_mean,masked_no_clahe_mean,delta_masked_minus_current,current_full_preprocessed_std,masked_no_clahe_std
0,auroc,0.946,0.971,0.024,0.036,0.019
1,f1,0.923,0.943,0.020,0.046,0.016
2,precision,0.935,0.951,0.017,0.047,0.027
3,recall,0.912,0.937,0.025,0.058,0.047
4,test_acc,0.899,0.925,0.026,0.060,0.018
5,train_acc,0.991,1.000,0.009,0.005,0.000


## 9. Full Region Ablation with Masked/No-CLAHE Inputs

This mirrors the Step 2.4 region-ablation grid, but keeps the `masked_no_clahe` pixel source from this investigation.

The tested combinations are:

- single regions: `full`, `inner`, `boundary`, `context`
- `full+X` pairs: `full+inner`, `full+boundary`, `full+context`
- leave-one-out triples: `drop full`, `drop inner`, `drop boundary`, `drop context`
- all four regions

This is still contained inside Step 2.6 and writes only to `data/outputs/original_image_patch_investigation`.

In [11]:
RA_C_PER = X_ra_masked.shape[1] // len(RA_REGIONS)
RA_REGION_SLICES = {r: slice(i * RA_C_PER, (i + 1) * RA_C_PER) for i, r in enumerate(RA_REGIONS)}

RA_ABLATION_SPECS = [
    ("full", "single", ["full"]),
    ("inner", "single", ["inner"]),
    ("boundary", "single", ["boundary"]),
    ("context", "single", ["context"]),
    ("full+inner", "full+X", ["full", "inner"]),
    ("full+boundary", "full+X", ["full", "boundary"]),
    ("full+context", "full+X", ["full", "context"]),
    ("drop full", "leave-one-out", ["inner", "boundary", "context"]),
    ("drop inner", "leave-one-out", ["full", "boundary", "context"]),
    ("drop boundary", "leave-one-out", ["full", "inner", "context"]),
    ("drop context", "leave-one-out", ["full", "inner", "boundary"]),
    ("ALL four", "all", ["full", "inner", "boundary", "context"]),
]


def subset_ra_regions(X, regs):
    return np.concatenate([X[:, RA_REGION_SLICES[r], :, :] for r in regs], axis=1).astype(np.float32)


def train_ra_subset_one_seed(X_subset, y, groups, seed, combo, kind, regs, epochs=60):
    set_seed(seed)
    train_mask, test_mask = ra_stratified_group_holdout(groups, y, seed)
    Xn = ra_normalise_train_only(X_subset, train_mask)
    Xtr = torch.tensor(Xn[train_mask], dtype=torch.float32)
    ytr = torch.tensor(y[train_mask], dtype=torch.long)
    Xte = torch.tensor(Xn[test_mask], dtype=torch.float32)

    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=24, shuffle=True)
    model = RAWSMapCNN(X_subset.shape[1]).to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=3e-4)
    loss_fn = nn.CrossEntropyLoss()

    start = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimiser.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimiser.step()

    model.eval()
    with torch.no_grad():
        train_prob = torch.softmax(model(Xtr.to(DEVICE)).cpu(), dim=1)[:, 1].numpy()
        test_prob = torch.softmax(model(Xte.to(DEVICE)).cpu(), dim=1)[:, 1].numpy()

    y_train_np = y[train_mask]
    y_test_np = y[test_mask]
    train_pred = (train_prob >= 0.5).astype(int)
    test_pred = (test_prob >= 0.5).astype(int)

    return {
        "combo": combo,
        "kind": kind,
        "regions": "+".join(regs),
        "pixel_source": RA_PIXEL_SOURCE,
        "seed": seed,
        "n_regions": len(regs),
        "n_channels": int(X_subset.shape[1]),
        "n_train": int(train_mask.sum()),
        "n_test": int(test_mask.sum()),
        "train_acc": accuracy_score(y_train_np, train_pred),
        "test_acc": accuracy_score(y_test_np, test_pred),
        "precision": precision_score(y_test_np, test_pred, zero_division=0),
        "recall": recall_score(y_test_np, test_pred, zero_division=0),
        "f1": f1_score(y_test_np, test_pred, zero_division=0),
        "auroc": roc_auc_score(y_test_np, test_prob) if len(np.unique(y_test_np)) == 2 else np.nan,
        "confusion_matrix": confusion_matrix(y_test_np, test_pred).tolist(),
        "seconds": time.time() - start,
    }


ra_ablation_perseed_path = OUT / "region_ablation_masked_no_clahe_cnnhead_per_seed.csv"
if ra_ablation_perseed_path.exists():
    ra_ablation_per_seed = pd.read_csv(ra_ablation_perseed_path)
    completed_ablation = set(zip(ra_ablation_per_seed["combo"], ra_ablation_per_seed["seed"].astype(int)))
    ablation_rows = ra_ablation_per_seed.to_dict("records")
else:
    completed_ablation = set()
    ablation_rows = []

for combo, kind, regs in RA_ABLATION_SPECS:
    X_subset = subset_ra_regions(X_ra_masked, regs)
    for seed in SEEDS:
        key = (combo, int(seed))
        if key in completed_ablation:
            print(f"Skipping completed masked/no-CLAHE ablation {combo} seed {seed}")
            continue
        print(f"Running masked/no-CLAHE ablation {combo} seed {seed} ({len(regs)} regions, {X_subset.shape[1]} channels)")
        result = train_ra_subset_one_seed(X_subset, y_ra_masked, groups_ra_masked, seed, combo, kind, regs)
        ablation_rows.append(result)
        pd.DataFrame(ablation_rows).to_csv(ra_ablation_perseed_path, index=False)
        print({k: result[k] for k in ["combo", "seed", "test_acc", "auroc", "f1"]})

ra_ablation_per_seed = pd.DataFrame(ablation_rows)
ra_ablation_per_seed.to_csv(ra_ablation_perseed_path, index=False)

ra_ablation_summary = (
    ra_ablation_per_seed
    .groupby(["combo", "kind", "regions", "pixel_source", "n_regions", "n_channels"], as_index=False)
    .agg(
        auroc=("auroc", "mean"),
        auroc_sd=("auroc", "std"),
        test_acc=("test_acc", "mean"),
        test_acc_sd=("test_acc", "std"),
        f1=("f1", "mean"),
        f1_sd=("f1", "std"),
        precision=("precision", "mean"),
        recall=("recall", "mean"),
        train_acc=("train_acc", "mean"),
        n_seeds=("seed", "nunique"),
    )
    .sort_values(["auroc", "test_acc"], ascending=[False, False])
)
ra_ablation_summary.to_csv(OUT / "region_ablation_masked_no_clahe_cnnhead_results.csv", index=False)

current_ablation = pd.read_csv(DATA / "outputs" / "region_ws_ablation" / "region_ablation_cnnhead_results.csv")
ra_ablation_comparison = current_ablation.merge(
    ra_ablation_summary,
    on=["combo", "kind"],
    how="inner",
    suffixes=("_current_full_preprocessed", "_masked_no_clahe"),
)
ra_ablation_comparison["auroc_delta_masked_minus_current"] = ra_ablation_comparison["auroc_masked_no_clahe"] - ra_ablation_comparison["auroc_current_full_preprocessed"]
ra_ablation_comparison["test_acc_delta_masked_minus_current"] = ra_ablation_comparison["test_acc_masked_no_clahe"] - ra_ablation_comparison["test_acc_current_full_preprocessed"]
ra_ablation_comparison["f1_delta_masked_minus_current"] = ra_ablation_comparison["f1_masked_no_clahe"] - ra_ablation_comparison["f1_current_full_preprocessed"]
ra_ablation_comparison = ra_ablation_comparison.sort_values(["auroc_masked_no_clahe", "test_acc_masked_no_clahe"], ascending=[False, False])
ra_ablation_comparison.to_csv(OUT / "region_ablation_masked_no_clahe_vs_current_comparison.csv", index=False)

print("Masked/no-CLAHE region ablation summary")
display(ra_ablation_summary.round(3))
print("Masked/no-CLAHE vs current full-preprocessed ablation comparison")
display(ra_ablation_comparison[[
    "combo", "kind", "auroc_current_full_preprocessed", "auroc_masked_no_clahe", "auroc_delta_masked_minus_current",
    "test_acc_current_full_preprocessed", "test_acc_masked_no_clahe", "test_acc_delta_masked_minus_current",
    "f1_current_full_preprocessed", "f1_masked_no_clahe", "f1_delta_masked_minus_current",
]].round(3))

Running masked/no-CLAHE ablation full seed 1 (1 regions, 91 channels)


{'combo': 'full', 'seed': 1, 'test_acc': 0.8461538461538461, 'auroc': 0.90625, 'f1': 0.875}
Running masked/no-CLAHE ablation full seed 7 (1 regions, 91 channels)


{'combo': 'full', 'seed': 7, 'test_acc': 0.7727272727272727, 'auroc': 0.9714285714285714, 'f1': 0.8}
Running masked/no-CLAHE ablation full seed 34 (1 regions, 91 channels)


{'combo': 'full', 'seed': 34, 'test_acc': 0.88, 'auroc': 0.9779411764705882, 'f1': 0.9090909090909091}
Running masked/no-CLAHE ablation full seed 42 (1 regions, 91 channels)


{'combo': 'full', 'seed': 42, 'test_acc': 0.9565217391304348, 'auroc': 0.9732142857142856, 'f1': 0.967741935483871}
Running masked/no-CLAHE ablation full seed 99 (1 regions, 91 channels)


...

{'combo': 'ALL four', 'seed': 99, 'test_acc': 0.9166666666666666, 'auroc': 0.9609375, 'f1': 0.9375}


Masked/no-CLAHE region ablation summary


,combo,kind,regions,pixel_source,n_regions,n_channels,auroc,auroc_sd,test_acc,test_acc_sd,f1,f1_sd,precision,recall,train_acc,n_seeds
3,drop boundary,leave-one-out,full+inner+context,masked_no_clahe,3,273,0.986,0.014,0.900,0.038,0.922,0.032,0.951,0.898,0.998,5
6,drop inner,leave-one-out,full+boundary+context,masked_no_clahe,3,273,0.977,0.009,0.908,0.019,0.930,0.015,0.950,0.912,0.998,5
9,full+context,full+X,full+context,masked_no_clahe,2,182,0.974,0.032,0.891,0.052,0.911,0.046,0.976,0.861,0.996,5
4,drop context,leave-one-out,full+inner+boundary,masked_no_clahe,3,273,0.973,0.008,0.908,0.019,0.931,0.013,0.940,0.924,0.996,5
0,ALL four,all,full+inner+boundary+context,masked_no_clahe,4,364,0.971,0.019,0.925,0.018,0.943,0.016,0.951,0.937,1.000,5
10,full+inner,full+X,full+inner,masked_no_clahe,2,182,0.966,0.031,0.909,0.045,0.930,0.035,0.938,0.925,1.000,5
7,full,single,full,masked_no_clahe,1,91,0.960,0.030,0.858,0.067,0.884,0.061,0.948,0.835,0.996,5
8,full+boundary,full+X,full+boundary,masked_no_clahe,2,182,0.957,0.018,0.900,0.037,0.921,0.031,0.964,0.887,0.996,5
11,inner,single,inner,masked_no_clahe,1,91,0.909,0.036,0.806,0.090,0.846,0.091,0.873,0.837,0.919,5
5,drop full,leave-one-out,inner+boundary+context,masked_no_clahe,3,273,0.908,0.052,0.825,0.052,0.869,0.048,0.856,0.887,0.941,5


Masked/no-CLAHE vs current full-preprocessed ablation comparison


,combo,kind,auroc_current_full_preprocessed,auroc_masked_no_clahe,auroc_delta_masked_minus_current,test_acc_current_full_preprocessed,test_acc_masked_no_clahe,test_acc_delta_masked_minus_current,f1_current_full_preprocessed,f1_masked_no_clahe,f1_delta_masked_minus_current
9,drop boundary,leave-one-out,0.939,0.986,0.047,0.858,0.900,0.042,0.893,0.922,0.029
8,drop inner,leave-one-out,0.926,0.977,0.052,0.834,0.908,0.074,0.876,0.930,0.053
6,full+context,full+X,0.918,0.974,0.057,0.801,0.891,0.090,0.850,0.911,0.062
10,drop context,leave-one-out,0.931,0.973,0.042,0.865,0.908,0.043,0.900,0.931,0.030
11,ALL four,all,0.946,0.971,0.024,0.899,0.925,0.026,0.923,0.943,0.020
4,full+inner,full+X,0.940,0.966,0.026,0.874,0.909,0.035,0.907,0.930,0.024
0,full,single,0.904,0.960,0.055,0.800,0.858,0.057,0.840,0.884,0.044
5,full+boundary,full+X,0.933,0.957,0.025,0.835,0.900,0.065,0.871,0.921,0.050
1,inner,single,0.922,0.909,-0.013,0.865,0.806,-0.059,0.898,0.846,-0.051
7,drop full,leave-one-out,0.928,0.908,-0.020,0.856,0.825,-0.031,0.889,0.869,-0.021
